In [1]:
import numpy as np
import torch

In [2]:
words = open('names.txt','r').read().splitlines()

In [3]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [4]:
#bi-gram vectors char to int & int to char
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [44]:
itos[5],stoi['e']

('e', 5)

tri-gram model sampling via counting method

In [6]:
#all possible bi-gram combinations (27 x 27 -> 729)

i=0
bigramtoi={}
bigram_chars = ['.']+ chars
for c1 in bigram_chars:
    for c2 in bigram_chars:
        s=c1+c2
        bigramtoi[s] =i
        i+= 1
bigramtoi

{'..': 0,
 '.a': 1,
 '.b': 2,
 '.c': 3,
 '.d': 4,
 '.e': 5,
 '.f': 6,
 '.g': 7,
 '.h': 8,
 '.i': 9,
 '.j': 10,
 '.k': 11,
 '.l': 12,
 '.m': 13,
 '.n': 14,
 '.o': 15,
 '.p': 16,
 '.q': 17,
 '.r': 18,
 '.s': 19,
 '.t': 20,
 '.u': 21,
 '.v': 22,
 '.w': 23,
 '.x': 24,
 '.y': 25,
 '.z': 26,
 'a.': 27,
 'aa': 28,
 'ab': 29,
 'ac': 30,
 'ad': 31,
 'ae': 32,
 'af': 33,
 'ag': 34,
 'ah': 35,
 'ai': 36,
 'aj': 37,
 'ak': 38,
 'al': 39,
 'am': 40,
 'an': 41,
 'ao': 42,
 'ap': 43,
 'aq': 44,
 'ar': 45,
 'as': 46,
 'at': 47,
 'au': 48,
 'av': 49,
 'aw': 50,
 'ax': 51,
 'ay': 52,
 'az': 53,
 'b.': 54,
 'ba': 55,
 'bb': 56,
 'bc': 57,
 'bd': 58,
 'be': 59,
 'bf': 60,
 'bg': 61,
 'bh': 62,
 'bi': 63,
 'bj': 64,
 'bk': 65,
 'bl': 66,
 'bm': 67,
 'bn': 68,
 'bo': 69,
 'bp': 70,
 'bq': 71,
 'br': 72,
 'bs': 73,
 'bt': 74,
 'bu': 75,
 'bv': 76,
 'bw': 77,
 'bx': 78,
 'by': 79,
 'bz': 80,
 'c.': 81,
 'ca': 82,
 'cb': 83,
 'cc': 84,
 'cd': 85,
 'ce': 86,
 'cf': 87,
 'cg': 88,
 'ch': 89,
 'ci': 90,
 'cj': 91

In [7]:
itobigram = {i:s for s,i in bigramtoi.items()}
itobigram


{0: '..',
 1: '.a',
 2: '.b',
 3: '.c',
 4: '.d',
 5: '.e',
 6: '.f',
 7: '.g',
 8: '.h',
 9: '.i',
 10: '.j',
 11: '.k',
 12: '.l',
 13: '.m',
 14: '.n',
 15: '.o',
 16: '.p',
 17: '.q',
 18: '.r',
 19: '.s',
 20: '.t',
 21: '.u',
 22: '.v',
 23: '.w',
 24: '.x',
 25: '.y',
 26: '.z',
 27: 'a.',
 28: 'aa',
 29: 'ab',
 30: 'ac',
 31: 'ad',
 32: 'ae',
 33: 'af',
 34: 'ag',
 35: 'ah',
 36: 'ai',
 37: 'aj',
 38: 'ak',
 39: 'al',
 40: 'am',
 41: 'an',
 42: 'ao',
 43: 'ap',
 44: 'aq',
 45: 'ar',
 46: 'as',
 47: 'at',
 48: 'au',
 49: 'av',
 50: 'aw',
 51: 'ax',
 52: 'ay',
 53: 'az',
 54: 'b.',
 55: 'ba',
 56: 'bb',
 57: 'bc',
 58: 'bd',
 59: 'be',
 60: 'bf',
 61: 'bg',
 62: 'bh',
 63: 'bi',
 64: 'bj',
 65: 'bk',
 66: 'bl',
 67: 'bm',
 68: 'bn',
 69: 'bo',
 70: 'bp',
 71: 'bq',
 72: 'br',
 73: 'bs',
 74: 'bt',
 75: 'bu',
 76: 'bv',
 77: 'bw',
 78: 'bx',
 79: 'by',
 80: 'bz',
 81: 'c.',
 82: 'ca',
 83: 'cb',
 84: 'cc',
 85: 'cd',
 86: 'ce',
 87: 'cf',
 88: 'cg',
 89: 'ch',
 90: 'ci',
 91: 'cj'

In [8]:
T= torch.zeros((729,27), dtype=torch.int32) # int works for count instead of float

In [9]:
#Tri-gram frequency count
for w in words:
    ws = ['.']+['.'] + list(w) + ['.']
    for c1,c2,c3 in zip(ws,ws[1:],ws[2:]):
        s= c1+c2
        id1 = bigramtoi[s]
        id2 = stoi[c3]
        T[id1,id2] += 1 

In [10]:
T.shape

torch.Size([729, 27])

In [11]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(729,729))
plt.imshow(T, cmap='Blues')
for i in range(729):
    for j in range(27):
        chstr = itobigram[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, T[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [12]:
P = T.float()
P /= P.sum(1, keepdims=True)

In [13]:
P.shape

torch.Size([729, 27])

tri-gram probability sampling via counting method

In [54]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    ix=0
    out='..'
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out +=itos[ix]
        if ix == 0:
            break
        id = out[-2:]
        ix = bigramtoi[id]
    print(out.strip('.'))

ce
bra
jalius
rochityharlonimittain
luwak
ka
da
samiyah
javer
gotai


In [ ]:
# GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling)
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood
# equivalent to minimizing the average negative log likelihood

# log(a*b*c) = log(a) + log(b) + log(c)

negative log likelihood bi-gram vs tri-gram counting method

![alt text](nll.png "Negative log likelihood function")


In [17]:
log_likelihood_tri = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  ws = ['.'] +['.']+ list(w) + ['.']
  for c1, c2,c3 in zip(ws, ws[1:],ws[2:]):
    s= c1+c2
    ix1 = bigramtoi[s]
    ix2 = stoi[c3]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood_tri += logprob
    n += 1
    #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelihood_tri=}')
nll = -log_likelihood_tri
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood_tri=tensor(-498647.7812)
nll=tensor(498647.7812)
2.185652017593384


Neural Nets Method tri-gram

In [25]:
# create the training set of bigrams (x,y)
# X input char Y output char

xs, ys = [], []

for w in words[:1]:
  ws = ['.']+ ['.'] + list(w) + ['.']
  for c1, c2,c3 in zip(ws, ws[1:],ws[2:]):
    s = c1+c2
    ix1 = bigramtoi[s]
    ix2 = stoi[c3]
    print(s, c3)
    xs.append(ix1)
    ys.append(ix2)
    
xs = torch.tensor(xs)
ys = torch.tensor(ys)

.. e
.e m
em m
mm a
ma .


In [26]:
xs

tensor([  0,   5, 148, 364, 352])

In [27]:
ys

tensor([ 5, 13, 13,  1,  0])

In [29]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=729).float()
xenc

tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [30]:
xenc.shape

torch.Size([5, 729])

In [32]:
W =torch.randn((729,27))

In [33]:
W.shape

torch.Size([729, 27])

In [34]:
xenc @ W

tensor([[ 5.4096e-01,  5.4270e-02, -8.0982e-01,  5.7480e-01,  2.4132e-02,
         -2.1565e+00,  4.2764e-01, -1.1243e-01, -2.4591e-01,  1.3502e+00,
          9.7975e-02,  1.3964e+00,  8.2629e-01,  2.1459e-01,  1.2790e+00,
         -1.1714e+00,  1.4338e-01, -2.1394e-01,  7.5797e-01,  5.8477e-01,
          1.8244e+00,  4.5343e-01, -5.5022e-01, -6.3149e-01,  5.6043e-01,
         -1.4951e+00,  9.2701e-01],
        [-4.0399e-01,  1.5813e+00,  5.7199e-01, -8.0874e-01,  1.2831e+00,
          1.3672e-01, -5.9954e-01,  3.5769e-02, -4.5317e-01, -1.8611e-01,
          7.7557e-01, -7.2328e-04,  9.6638e-01,  1.2528e+00, -4.0323e-01,
          5.1592e-01,  8.1726e-01,  3.9131e-01, -1.6763e+00,  4.6459e-01,
          1.0735e+00,  3.2030e-01,  8.9213e-01, -1.1090e+00, -4.4300e-01,
         -7.2648e-01,  1.5588e+00],
        [ 6.6679e-01, -5.8590e-01, -1.4805e-01, -1.5410e+00,  1.1255e-01,
         -2.5254e-01, -1.8445e+00, -5.4385e-01,  2.5500e-01,  1.1911e-02,
         -1.0659e+00, -1.3852e+00, -1.53

In [35]:
logits = xenc @ W
counts = logits.exp() # e ^ (wx)
probs = counts /counts.sum(1,keepdims = True)
probs

tensor([[0.0381, 0.0234, 0.0099, 0.0394, 0.0227, 0.0026, 0.0340, 0.0198, 0.0174,
         0.0856, 0.0245, 0.0897, 0.0507, 0.0275, 0.0798, 0.0069, 0.0256, 0.0179,
         0.0474, 0.0398, 0.1376, 0.0349, 0.0128, 0.0118, 0.0389, 0.0050, 0.0561],
        [0.0146, 0.1064, 0.0388, 0.0098, 0.0790, 0.0251, 0.0120, 0.0227, 0.0139,
         0.0182, 0.0476, 0.0219, 0.0576, 0.0766, 0.0146, 0.0367, 0.0496, 0.0324,
         0.0041, 0.0348, 0.0641, 0.0302, 0.0534, 0.0072, 0.0141, 0.0106, 0.1041],
        [0.0418, 0.0120, 0.0185, 0.0046, 0.0240, 0.0167, 0.0034, 0.0125, 0.0277,
         0.0217, 0.0074, 0.0054, 0.0046, 0.0138, 0.0232, 0.0108, 0.1298, 0.0219,
         0.1023, 0.0204, 0.0274, 0.1995, 0.1573, 0.0152, 0.0127, 0.0367, 0.0287],
        [0.0135, 0.1055, 0.0180, 0.0349, 0.0977, 0.0428, 0.0535, 0.0642, 0.0047,
         0.0349, 0.0801, 0.0050, 0.0028, 0.1285, 0.0452, 0.0031, 0.0271, 0.0090,
         0.0493, 0.0097, 0.0146, 0.0076, 0.0305, 0.0155, 0.0594, 0.0050, 0.0379],
        [0.0034, 0.0138,

In [36]:
#probs[torch.arange(5),ys] -> picks out the pobability value from the forward pass for the actual output in the training


loss = -probs[torch.arange(5),ys].log().mean() 



In [37]:
print(loss.item())

4.1473588943481445


In [38]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g)

In [39]:
xenc = F.one_hot(xs, num_classes=729).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax'

In [40]:
probs.shape

torch.Size([5, 27])

In [ ]:
probs[0,5]

tensor([[0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
         0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
         0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459],
        [0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
         0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
         0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472],
        [0.0225, 0.1182, 0.0491, 0.0079, 0.0210, 0.0090, 0.0082, 0.0792, 0.0857,
         0.0670, 0.0166, 0.0229, 0.0127, 0.0082, 0.1269, 0.0384, 0.0237, 0.0041,
         0.0257, 0.0761, 0.0642, 0.0330, 0.0047, 0.0161, 0.0190, 0.0322, 0.0077],
        [0.0749, 0.0326, 0.0100, 0.0488, 0.0360, 0.0102, 0.0430, 0.0246, 0.0238,
         0.0511, 0.0037, 0.0019, 0.0767, 0.0118, 0.0222, 0.0137, 0.0130, 0.0087,
         0.0104, 0.0319, 0.0474, 0.0094, 0.0037, 0.2830, 0.0036, 0.0918, 0.0120],
        [0.0108, 0.0381,

In [42]:
nlls = torch.zeros(5)
for i in range(5):
  # i-th bigram:
  x = xs[i].item() # input character index
  y = ys[i].item() # label character index
  print('--------')
  print(f'trigram example {i+1}: {itobigram[x]} -> {itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

--------
trigram example 1: .. -> e (indexes 0,5)
input to the neural net: 0
output probabilities from the neural net: tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459])
label (actual next character): 5
probability assigned by the net to the the correct character: 0.012286250479519367
log likelihood: -4.3992743492126465
negative log likelihood: 4.3992743492126465
--------
trigram example 2: .e -> m (indexes 5,13)
input to the neural net: 5
output probabilities from the neural net: tensor([0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
        0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
        0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472])
label (actual next character): 13
probability assigned by the net to the the correc

In [45]:
# create the dataset
xs, ys = [], []
for w in words:
  ws = ['.'] + ['.'] + list(w) + ['.']
  for c1, c2,c3 in zip(ws, ws[1:],ws[2:]):
    s = c1+c2
    ix1 = bigramtoi[s]
    ix2 = stoi[c3]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

number of examples:  228146


In [49]:
# gradient descent
for k in range(100):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=729).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -100 * W.grad

2.426764726638794
2.424663782119751
2.4226043224334717
2.4205844402313232
2.4186034202575684
2.4166598320007324
2.4147534370422363
2.41288161277771
2.4110445976257324
2.40924072265625
2.4074692726135254
2.4057295322418213
2.4040212631225586
2.4023420810699463
2.4006919860839844
2.3990702629089355
2.3974759578704834
2.3959083557128906
2.394367218017578
2.3928511142730713
2.391359806060791
2.389892339706421
2.388448715209961
2.3870277404785156
2.385629177093506
2.3842523097991943
2.382896661758423
2.381561756134033
2.380247116088867
2.3789522647857666
2.377676486968994
2.37641978263855
2.375181198120117
2.373960494995117
2.372757911682129
2.3715720176696777
2.3704030513763428
2.369250535964966
2.3681137561798096
2.3669931888580322
2.3658878803253174
2.364797353744507
2.3637216091156006
2.3626608848571777
2.3616137504577637
2.3605799674987793
2.35956072807312
2.3585543632507324
2.3575611114501953
2.3565804958343506
2.3556125164031982
2.3546571731567383
2.353713274002075
2.3527817726135254

In [55]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
  
  out = '..'
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:

    tok = bigramtoi[out[-2:]]
    xenc = F.one_hot(torch.tensor([tok]), num_classes=729).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out += itos[ix]
    if ix == 0:
      break
  print(out.strip('.'))

cexzdfzjglkuriana
kayhhmvlzimilea
noluwan
ka
da
samiyah
javer
gitai
moriellavojkwpteda
kaley


Split training dataset into train, test and dev (80%,10%,10%)

In [64]:
from torch.utils.data import TensorDataset, random_split

In [58]:
train,test,dev = random_split(range(30), [0.8, 0.1, 0.1], generator=g)

In [63]:
indices = torch.randperm(num)
indices

tensor([ 27038, 210660,  89627,  ..., 139889, 118115, 165968])

In [ ]:
# create the dataset
xs, ys = [], []
for w in words:
  ws = ['.'] + ['.'] + list(w) + ['.']
  for c1, c2,c3 in zip(ws, ws[1:],ws[2:]):
    s = c1+c2
    ix1 = bigramtoi[s]
    ix2 = stoi[c3]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

In [66]:
xs.shape

torch.Size([228146])

In [67]:
# Create a TensorDataset
dataset = TensorDataset(xs, ys)

# Calculate split sizes (80%, 10%, 10%)
train_size = int(0.8 * num)
dev_size = int(0.1 * num)
test_size = num - train_size - dev_size  # remainder to handle rounding

# Split the dataset
train_dataset, dev_dataset, test_dataset = random_split(
    dataset, 
    [train_size, dev_size, test_size]
)

print(f'Train: {len(train_dataset)} examples')
print(f'Dev: {len(dev_dataset)} examples')
print(f'Test: {len(test_dataset)} examples')



Train: 182516 examples
Dev: 22814 examples
Test: 22816 examples


In [113]:
train_indices = train_dataset.indices
xtr, ytr = xs[train_indices], ys[train_indices]
num_train = xtr.nelement()

In [110]:
xtr.shape

torch.Size([182516])

In [111]:
ytr.shape

torch.Size([182516])

In [114]:
num_train

182516

In [142]:
# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [143]:
# gradient descent
for k in range(100):
  
  # forward pass
  xenc = F.one_hot(xtr, num_classes=729).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num_train), ytr].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -100 * W.grad

3.802034378051758
3.5466299057006836
3.426182508468628
3.3341147899627686
3.257277250289917
3.192277669906616
3.136435031890869
3.087763547897339
3.0446789264678955
3.00610613822937
2.971285343170166
2.9396727085113525
2.9108481407165527
2.8844757080078125
2.860273599624634
2.838003396987915
2.817456007003784
2.798447847366333
2.7808139324188232
2.7644076347351074
2.749095916748047
2.7347631454467773
2.7213053703308105
2.7086329460144043
2.696667432785034
2.6853413581848145
2.6745948791503906
2.6643784046173096
2.654646158218384
2.645360231399536
2.6364855766296387
2.627992630004883
2.6198537349700928
2.6120448112487793
2.604543685913086
2.5973315238952637
2.5903894901275635
2.5837018489837646
2.577253818511963
2.5710318088531494
2.565023422241211
2.5592167377471924
2.5536017417907715
2.548168420791626
2.542907476425171
2.537811040878296
2.5328710079193115
2.5280802249908447
2.523430824279785
2.5189173221588135
2.514533519744873
2.5102741718292236
2.5061333179473877
2.502106189727783
2

In [109]:
ytr.shape

torch.Size([182516])

In [137]:
# Train & Dev Loss


# Train Loss
xenc = F.one_hot(xtr, num_classes=729).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(num_train), ytr].log().mean() + 0.001*(W**2).mean()
print(f'Training Loss : {loss.item()}')

# Dev Loss

xenc = F.one_hot(xdev, num_classes=729).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(num_dev), ydev].log().mean()
print(f'Dev Loss : {loss.item()}')
  

Training Loss : 2.379399299621582
Dev Loss : 2.3973841667175293


In [144]:
# Test Loss
test_indices = test_dataset.indices
xte, yte = xs[test_indices], ys[test_indices]
num_test = xte.nelement()

xenc = F.one_hot(xte, num_classes=729).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(num_test), yte].log().mean()
print(f'Test Loss : {loss.item()}')

Test Loss : 2.400712490081787


alpha -> 1

Training Loss : 2.658615827560425
Dev Loss : 2.437021493911743

alpha -> 0.1

Training Loss : 2.4654481410980225
Dev Loss : 2.3965282440185547
Test Loss : 2.3998072147369385

alpha -> 0.01

Training Loss : 2.3884596824645996
Dev Loss : 2.3972294330596924
Test Loss : 2.400712490081787

alpha -> 0.001

Training Loss : 2.379399299621582
Dev Loss : 2.3973841667175293


In [140]:
test_indices = test_dataset.indices
xte, yte = xs[test_indices], ys[test_indices]
num_test = xte.nelement()

In [99]:
correct_preds = 0
for x,y in test_dataset:
    x_val = F.one_hot(torch.tensor([x]), num_classes=729).float()
    logits = x_val @ W 
    counts = logits.exp()
    p = counts / counts.sum(1, keepdims=True)
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    if y.item() == ix:
        correct_preds += 1

In [101]:
test_accuracy = correct_preds / num_test
print(f'Test accuracy : {test_accuracy} -> Correct preds : {correct_preds} , Test samples : {num_test}')

Test accuracy : 0.16072054698457222 -> Correct preds : 3667 , Test samples : 22816


In [116]:
dev_indices = dev_dataset.indices
xdev, ydev = xs[dev_indices], ys[dev_indices]
num_dev = xdev.nelement()

In [102]:
correct_preds = 0
for x,y in dev_dataset:
    x_val = F.one_hot(torch.tensor([x]), num_classes=729).float()
    logits = x_val @ W 
    counts = logits.exp()
    p = counts / counts.sum(1, keepdims=True)
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    if y.item() == ix:
        correct_preds += 1

In [105]:
dev_accuracy = correct_preds / num_dev
print(f'Dev accuracy : {dev_accuracy} -> Correct preds : {correct_preds} , dev samples : {num_dev}')

Dev accuracy : 0.15814850530376084 -> Correct preds : 3608 , dev samples : 22814


Replacing Matrix Multiplication with indexing for efficiency

Since one-hot encoding provides just a single 1 value and 728 zero's

In [145]:
# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [156]:
# gradient descent
for k in range(100):
  
  # forward pass
  #xenc = F.one_hot(xtr, num_classes=729).float() # input to the network: one-hot encoding
  logits = W[xtr] # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num_train), ytr].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -100 * W.grad

3.802034378051758
3.546630620956421
3.426182508468628
3.3341152667999268
3.257277250289917
3.192277669906616
3.136435031890869
3.087763547897339
3.0446789264678955
3.00610613822937
2.971285343170166
2.9396727085113525
2.9108481407165527
2.8844757080078125
2.860273838043213
2.838003396987915
2.817456007003784
2.798447847366333
2.7808139324188232
2.7644076347351074
2.749095916748047
2.7347631454467773
2.7213053703308105
2.7086329460144043
2.696667432785034
2.6853408813476562
2.6745948791503906
2.6643781661987305
2.654646158218384
2.645360231399536
2.6364858150482178
2.627992630004883
2.6198537349700928
2.6120448112487793
2.604543685913086
2.5973315238952637
2.5903894901275635
2.5837018489837646
2.577253818511963
2.5710318088531494
2.565023422241211
2.5592167377471924
2.5536017417907715
2.548168420791626
2.542907476425171
2.537811040878296
2.5328710079193115
2.5280799865722656
2.523430824279785
2.5189175605773926
2.514533519744873
2.5102741718292236
2.5061333179473877
2.502106189727783
2.

In [150]:
xtr

tensor([ 52, 329, 333,  ...,   0, 622,   0])

tensor([-0.7125,  0.6541,  0.8071, -0.8077,  0.9841,  1.6233,  0.7369,  0.5528,
        -0.1671, -1.2738, -0.6363, -0.0440, -0.1868,  0.5433, -0.0905, -0.3554,
         0.9058, -0.5537, -0.3112,  0.5195, -0.7991, -0.7683,  0.9271,  0.2019,
        -1.1854,  1.0008,  0.9374], grad_fn=<SelectBackward0>)

In [155]:
W[52]

tensor([-0.7125,  0.6541,  0.8071, -0.8077,  0.9841,  1.6233,  0.7369,  0.5528,
        -0.1671, -1.2738, -0.6363, -0.0440, -0.1868,  0.5433, -0.0905, -0.3554,
         0.9058, -0.5537, -0.3112,  0.5195, -0.7991, -0.7683,  0.9271,  0.2019,
        -1.1854,  1.0008,  0.9374], grad_fn=<SelectBackward0>)